# Classification Model

**Problem**: [TODO: Problem description]
**Dataset**: [TODO: Dataset name]
**Author**: [TODO: Your name]
**Date**: [TODO: Date]

## Objective
[TODO: Describe the classification objective]

## 1. Setup

In [ ]:
# Parameters (for papermill parameterization)
SEED = 42
TEST_SIZE = 0.2
TARGET_COL = 'target'
DATA_PATH = 'data/raw/dataset.csv'

In [ ]:
# Standard library
import os
import sys
import random
from pathlib import Path
from datetime import datetime

# Data handling
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# ML
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier

# Set seeds for reproducibility
random.seed(SEED)
np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

# Display settings
pd.set_option('display.max_columns', 50)
plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

# Environment info
print(f"Timestamp: {datetime.now().isoformat()}")
print(f"Python: {sys.version}")
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")
print(f"Seed: {SEED}")

## 2. Data Loading

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df):,} rows, {len(df.columns)} columns")
df.head()

## 3. Quick EDA

In [ ]:
# Basic info
print(df.info())
print("\nMissing values:")
print(df.isnull().sum()[df.isnull().sum() > 0])

In [ ]:
# Target distribution
print(f"\nTarget distribution ({TARGET_COL}):")
print(df[TARGET_COL].value_counts())
print(f"\nClass balance:")
print(df[TARGET_COL].value_counts(normalize=True).round(3))

plt.figure(figsize=(8, 5))
df[TARGET_COL].value_counts().plot(kind='bar', edgecolor='black')
plt.title(f'Target Distribution: {TARGET_COL}')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

## 4. Data Preprocessing

In [ ]:
# Make a copy for processing
df_processed = df.copy()

# TODO: Handle missing values
# Example: df_processed = df_processed.dropna()
# Example: df_processed['col'] = df_processed['col'].fillna(df_processed['col'].median())

# TODO: Remove duplicates if needed
# df_processed = df_processed.drop_duplicates()

print(f"After preprocessing: {len(df_processed):,} rows")

## 5. Feature Engineering

In [ ]:
# TODO: Create new features
# Example: df_processed['feature_ratio'] = df_processed['a'] / (df_processed['b'] + 1)

# Encode categorical variables (if any)
categorical_cols = df_processed.select_dtypes(include=['object']).columns.tolist()
if TARGET_COL in categorical_cols:
    categorical_cols.remove(TARGET_COL)

print(f"Categorical columns to encode: {categorical_cols}")

# One-hot encode (for low cardinality)
if categorical_cols:
    df_processed = pd.get_dummies(df_processed, columns=categorical_cols, drop_first=True)

print(f"Final shape: {df_processed.shape}")

## 6. Train/Test Split

**CRITICAL**: All preprocessing that learns from data (scaling, encoding, imputation) must be fit on training data only!

In [ ]:
# Define features and target
FEATURE_COLS = [col for col in df_processed.columns if col != TARGET_COL]

X = df_processed[FEATURE_COLS]
y = df_processed[TARGET_COL]

# Encode target if categorical
if y.dtype == 'object':
    le = LabelEncoder()
    y = le.fit_transform(y)
    print(f"Target classes: {le.classes_}")

# Split BEFORE any scaling
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=y  # Maintain class distribution
)

print(f"Train size: {len(X_train):,} ({len(X_train)/len(X)*100:.1f}%)")
print(f"Test size: {len(X_test):,} ({len(X_test)/len(X)*100:.1f}%)")
print(f"\nTrain target distribution:")
print(pd.Series(y_train).value_counts(normalize=True).round(3))
print(f"\nTest target distribution:")
print(pd.Series(y_test).value_counts(normalize=True).round(3))

In [ ]:
# Scale features (fit on train only!)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)  # Transform only, no fit!

print("Scaling complete. Train mean ~0, std ~1")
print(f"Train mean: {X_train_scaled.mean():.6f}")
print(f"Train std: {X_train_scaled.std():.6f}")

## 7. Model Definition

In [ ]:
# Baseline model
baseline = DummyClassifier(strategy='most_frequent', random_state=SEED)
baseline.fit(X_train_scaled, y_train)
baseline_pred = baseline.predict(X_test_scaled)
baseline_acc = accuracy_score(y_test, baseline_pred)
print(f"Baseline accuracy (most frequent): {baseline_acc:.4f}")

In [ ]:
# Primary model
MODEL_PARAMS = {
    'n_estimators': 100,
    'max_depth': 10,
    'min_samples_split': 5,
    'min_samples_leaf': 2,
    'random_state': SEED,
    'n_jobs': -1
}

model = RandomForestClassifier(**MODEL_PARAMS)
print(f"Model: {model.__class__.__name__}")
print(f"Parameters: {MODEL_PARAMS}")

## 8. Training

In [ ]:
import time

# Train model
start_time = time.time()
model.fit(X_train_scaled, y_train)
train_time = time.time() - start_time
print(f"Training time: {train_time:.2f} seconds")

In [ ]:
# Cross-validation on training data
cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='accuracy')
print(f"CV Accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std()*2:.4f})")
print(f"CV Scores: {cv_scores.round(4)}")

## 9. Evaluation

In [ ]:
# Predictions
y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)

# For binary classification, get probability of positive class
if y_prob.shape[1] == 2:
    y_prob_pos = y_prob[:, 1]
else:
    y_prob_pos = y_prob

In [ ]:
# Classification report
print("Classification Report:")
print("=" * 60)
print(classification_report(y_test, y_pred))

In [ ]:
# Summary metrics
metrics = {
    'accuracy': accuracy_score(y_test, y_pred),
    'precision_weighted': precision_score(y_test, y_pred, average='weighted'),
    'recall_weighted': recall_score(y_test, y_pred, average='weighted'),
    'f1_weighted': f1_score(y_test, y_pred, average='weighted'),
}

# Add ROC AUC for binary classification
if len(np.unique(y_test)) == 2:
    metrics['roc_auc'] = roc_auc_score(y_test, y_prob_pos)

print("\nSummary Metrics:")
print("-" * 40)
for name, value in metrics.items():
    print(f"  {name}: {value:.4f}")

print(f"\nImprovement over baseline: {(metrics['accuracy'] - baseline_acc)*100:.2f}%")

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# ROC Curve (binary classification only)
if len(np.unique(y_test)) == 2:
    fpr, tpr, thresholds = roc_curve(y_test, y_prob_pos)
    
    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, label=f'Model (AUC = {metrics["roc_auc"]:.4f})')
    plt.plot([0, 1], [0, 1], 'k--', label='Random')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curve')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## 10. Error Analysis

In [ ]:
# Feature importance
importance_df = pd.DataFrame({
    'feature': FEATURE_COLS,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 8))
sns.barplot(data=importance_df.head(20), x='importance', y='feature')
plt.title('Top 20 Feature Importances')
plt.tight_layout()
plt.show()

print("Top 10 features:")
print(importance_df.head(10).to_string(index=False))

In [ ]:
# Analyze misclassifications
error_mask = y_pred != y_test
error_count = error_mask.sum()
error_rate = error_count / len(y_test)

print(f"Total errors: {error_count} ({error_rate*100:.2f}%)")
print(f"\nErrors by predicted class:")
print(pd.Series(y_pred[error_mask]).value_counts())
print(f"\nErrors by actual class:")
print(pd.Series(y_test[error_mask]).value_counts())

## 11. Save Artifacts

In [ ]:
import joblib
import json

# Create artifacts directory
ARTIFACTS_DIR = Path('artifacts')
ARTIFACTS_DIR.mkdir(exist_ok=True)

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

# Save model
model_path = ARTIFACTS_DIR / f'model_{timestamp}.joblib'
joblib.dump(model, model_path)
print(f"Model saved: {model_path}")

# Save scaler
scaler_path = ARTIFACTS_DIR / f'scaler_{timestamp}.joblib'
joblib.dump(scaler, scaler_path)
print(f"Scaler saved: {scaler_path}")

# Save metadata
metadata = {
    'timestamp': timestamp,
    'model_type': model.__class__.__name__,
    'model_params': MODEL_PARAMS,
    'feature_cols': FEATURE_COLS,
    'target_col': TARGET_COL,
    'metrics': metrics,
    'train_size': len(X_train),
    'test_size': len(X_test),
    'seed': SEED,
    'baseline_accuracy': baseline_acc,
    'cv_mean': float(cv_scores.mean()),
    'cv_std': float(cv_scores.std())
}

metadata_path = ARTIFACTS_DIR / f'metadata_{timestamp}.json'
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)
print(f"Metadata saved: {metadata_path}")

## 12. Conclusions

### Results Summary
- **Model**: [TODO: Model type]
- **Test Accuracy**: [TODO: X.XX]
- **Baseline**: [TODO: X.XX]
- **Improvement**: [TODO: X.XX%]

### Key Findings
1. [TODO: Most important features]
2. [TODO: Model performance observations]
3. [TODO: Error patterns]

### Limitations
- [TODO: Data limitations]
- [TODO: Model limitations]

### Next Steps
1. [TODO: Hyperparameter tuning]
2. [TODO: Try other models]
3. [TODO: Feature engineering]
4. [TODO: Deployment considerations]